# 03 — Modelos PyTorch

Serão comparados: DNN sobre TF-IDF, LSTM sobre tokens e Transformer Encoder pequeno treinado de raiz.

In [ ]:
import sys; sys.path.append('../')
import torch, pandas as pd, numpy as np
from torch.utils.data import TensorDataset, DataLoader
from src.models import TfidfVectorizerNumpy, build_torch_models, make_sequence_vocab, metrics
train=pd.read_csv('../data/processed/train.csv'); val=pd.read_csv('../data/processed/validation.csv'); test=pd.read_csv('../data/processed/test.csv')
device='cuda' if torch.cuda.is_available() else 'cpu'; print('Device:',device)

## 3.1 PyTorch DNN + TF-IDF

O TF-IDF é produzido pelo implementador NumPy; a rede neural é treinada com PyTorch.

In [ ]:
vec=TfidfVectorizerNumpy(max_features=3000).fit(train.Text)
Xtr=torch.tensor(vec.transform(train.Text)); Xv=torch.tensor(vec.transform(val.Text)); Xte=torch.tensor(vec.transform(test.Text))
ytr=torch.tensor(train.label.values,dtype=torch.float32); yv=torch.tensor(val.label.values,dtype=torch.float32); yte=test.label.values
TorchDNN, LSTMClassifier, TinyTransformer=build_torch_models(Xtr.shape[1])
model=TorchDNN(Xtr.shape[1]).to(device); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); lossfn=torch.nn.BCEWithLogitsLoss()
for ep in range(150):
    model.train(); opt.zero_grad(); loss=lossfn(model(Xtr.to(device)),ytr.to(device)); loss.backward(); opt.step()
model.eval(); pred=(torch.sigmoid(model(Xte.to(device))).detach().cpu().numpy()>=.5).astype(int)
print(metrics(yte,pred))

## 3.2 LSTM

Um vocabulário é criado **apenas com o treino**. Isso evita leakage do vocabulário a partir de validação/teste.

In [ ]:
vocab,encode=make_sequence_vocab(train.Text,max_vocab=3000,max_len=128)
A=lambda s: torch.tensor(np.stack([encode(x) for x in s]),dtype=torch.long)
Atr,Av,At=A(train.Text),A(val.Text),A(test.Text); ytr=torch.tensor(train.label.values,dtype=torch.float32)
model=LSTMClassifier(len(vocab)).to(device); opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
for ep in range(80):
    model.train(); opt.zero_grad(); loss=lossfn(model(Atr.to(device)),ytr.to(device)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
model.eval(); pred=(torch.sigmoid(model(At.to(device))).detach().cpu().numpy()>=.5).astype(int)
print(metrics(test.label.values,pred))

## 3.3 Transformer Encoder pequeno

Para manter o projeto executável mesmo sem depender de um modelo externo, usamos `torch.nn.TransformerEncoder` treinado de raiz. Uma alternativa posterior é fine-tuning de BERT/DistilBERT com Hugging Face Transformers, se houver GPU e dados suficientes.

In [ ]:
model=TinyTransformer(len(vocab),emb=64,nhead=4,layers=2).to(device); opt=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
for ep in range(80):
    model.train(); opt.zero_grad(); loss=lossfn(model(Atr.to(device)),ytr.to(device)); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
model.eval(); pred=(torch.sigmoid(model(At.to(device))).detach().cpu().numpy()>=.5).astype(int)
print(metrics(test.label.values,pred))